In [1]:
import os
import glob
import numpy as np
import pandas as pd
from tqdm import tqdm

In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:
dataset_path = "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/raw/UAH-DRIVESET-v1"

output_path = "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/processed"

In [4]:
def load_accelerometer(acc_path):

    acc_columns = [

        "timestamp",
        "active",

        "acc_x",
        "acc_y",
        "acc_z",

        "acc_x_kf",
        "acc_y_kf",
        "acc_z_kf",

        "roll",
        "pitch",
        "yaw"

    ]

    acc_df = pd.read_csv(
        acc_path,
        sep=r"\s+",
        header=None,
        names=acc_columns
    )

    return acc_df

def load_gps(gps_path):

    gps_columns = [

        "timestamp",
        "speed",
        "latitude",
        "longitude",
        "altitude",

        "gps_quality",
        "satellites",

        "heading",

        "extra_1",
        "extra_2",
        "extra_3",
        "extra_4"

    ]

    gps_df = pd.read_csv(
        gps_path,
        sep=r"\s+",
        header=None,
        names=gps_columns
    )

    return gps_df
def load_lane_detection(lane_path):

    lane_df = pd.read_csv(
        lane_path,
        sep=r"\s+",
        header=None
    )

    lane_df.columns = [

        "time",

        "lane_offset",

        "phi",

        "road_width",

        "lane_state"
    ]

    return lane_df

def clean_lane_detection(lane_df):

    lane_df = lane_df.copy()

    lane_df.replace(-9, np.nan, inplace=True)

    lane_df.replace(-99, np.nan, inplace=True)

    lane_df = lane_df.sort_values(
        "time"
    )

    lane_df = lane_df.reset_index(
        drop=True
    )

    return lane_df
def load_vehicle_detection(vehicle_path):

    vehicle_df = pd.read_csv(

        vehicle_path,

        sep=r"\s+",

        header=None

    )

    vehicle_df.columns = [

        "time",

        "front_distance",

        "relative_speed",

        "vehicle_state",

        "confidence"

    ]

    return vehicle_df

In [5]:
def synchronize_sensors(acc_df, gps_df):

    master_df = pd.merge_asof(

        acc_df.sort_values("timestamp"),

        gps_df.sort_values("timestamp"),

        on="timestamp",

        direction="nearest"

    )

    return master_df

def merge_lane_data(master_df, lane_df):

    master_df = master_df.copy()
    lane_df = lane_df.copy()

    master_df = master_df.sort_values(
        "timestamp"
    )

    lane_df = lane_df.sort_values(
        "time"
    )

    merged_df = pd.merge_asof(

        master_df,

        lane_df,

        left_on="timestamp",

        right_on="time",

        direction="nearest"

    )

    merged_df.drop(
        columns=["time"],
        inplace=True
    )

    return merged_df

def merge_vehicle_data(master_df, vehicle_df):

    master_df = master_df.copy()
    vehicle_df = vehicle_df.copy()

    master_df = master_df.sort_values(
        "timestamp"
    )

    vehicle_df = vehicle_df.sort_values(
        "time"
    )

    merged_df = pd.merge_asof(

        master_df,

        vehicle_df,

        left_on="timestamp",

        right_on="time",

        direction="nearest"

    )

    merged_df.drop(
        columns=["time"],
        inplace=True
    )

    return merged_df

def clean_vehicle_detection(vehicle_df):

    vehicle_df = vehicle_df.copy()

    vehicle_df = vehicle_df.sort_values(

        "time"

    )

    vehicle_df = vehicle_df.reset_index(

        drop=True

    )

    return vehicle_df

In [6]:
def engineer_features_v6(master_df):

    master_df = master_df.copy()

    # ----------------------------------------------------
    # Acceleration Features
    # ----------------------------------------------------

    master_df["acc_resultant"] = np.sqrt(
        master_df["acc_x"]**2 +
        master_df["acc_y"]**2 +
        master_df["acc_z"]**2
    )

    master_df["acc_horizontal"] = np.sqrt(
        master_df["acc_x"]**2 +
        master_df["acc_y"]**2
    )

    master_df["acc_vertical"] = master_df["acc_z"]

    # ----------------------------------------------------
    # Delta Features
    # ----------------------------------------------------

    master_df["speed_delta"] = master_df["speed"].diff().fillna(0)

    master_df["heading_delta"] = master_df["heading"].diff().fillna(0)

    master_df["roll_delta"] = master_df["roll"].diff().fillna(0)

    master_df["pitch_delta"] = master_df["pitch"].diff().fillna(0)

    master_df["yaw_delta"] = master_df["yaw"].diff().fillna(0)

    # ----------------------------------------------------
    # Lane Features
    # ----------------------------------------------------

    master_df["lane_departure"] = (
        master_df["lane_offset"].abs() > 0.75
    ).astype(int)

    master_df["steering_alignment"] = (
        master_df["lane_offset"] *
        master_df["phi"]
    )
   # Dynamic Lane Features

    master_df["lane_offset_velocity"] = (
        master_df["lane_offset"]
        .diff()
        .fillna(0)
    )
    # ------------------------------------------------
    # Vehicle Features
    # ------------------------------------------------

    master_df["vehicle_present"] = (
        master_df["front_distance"] > 0
    ).astype(int)

    master_df["safe_distance"] = np.where(
        master_df["vehicle_present"] == 1,
        master_df["front_distance"],
        np.nan
    )

    master_df["closing_speed"] = np.where(
        master_df["vehicle_present"] == 1,
        master_df["relative_speed"],
        0
    )

    master_df["ttc"] = np.where(
        (master_df["vehicle_present"] == 1) &
        (master_df["closing_speed"] > 0),
        master_df["front_distance"] /
        master_df["closing_speed"],
        np.nan
    )
    # ---------------------------------------------
    # Dynamic Vehicle Features
    # ---------------------------------------------

    master_df["ttc_trend"] = (
        master_df["ttc"]
        .diff()
        .fillna(0)
   )

    return master_df


In [7]:
def process_trip_v6(
    dataset_path,
    driver,
    trip,
    window_size=120
):

    trip_path = os.path.join(
        dataset_path,
        driver,
        trip
    )

    # -----------------------------
    # File Paths
    # -----------------------------

    acc_path = os.path.join(
        trip_path,
        "RAW_ACCELEROMETERS.txt"
    )

    gps_path = os.path.join(
        trip_path,
        "RAW_GPS.txt"
    )

    lane_path = os.path.join(
        trip_path,
        "PROC_LANE_DETECTION.txt"
    )

    vehicle_path = os.path.join(
        trip_path,
        "PROC_VEHICLE_DETECTION.txt"
    )

    # -----------------------------
    # Load
    # -----------------------------

    acc_df = load_accelerometer(acc_path)

    gps_df = load_gps(gps_path)

    lane_df = clean_lane_detection(
        load_lane_detection(lane_path)
    )

    vehicle_df = clean_vehicle_detection(
        load_vehicle_detection(vehicle_path)
    )

    # -----------------------------
    # Merge Sensors
    # -----------------------------

    master_df = synchronize_sensors(
        acc_df,
        gps_df
    )

    master_df = merge_lane_data(
        master_df,
        lane_df
    )

    master_df = merge_vehicle_data(
        master_df,
        vehicle_df
    )

    # -----------------------------
    # Feature Engineering
    # -----------------------------

    feature_df = engineer_features_v6(
        master_df
    )

    # -----------------------------
    # Sliding Windows
    # -----------------------------

    window_dataset = create_sliding_windows_v4(
        feature_df,
        window_features_v6,
        window_size
    )

    # -----------------------------
    # Metadata
    # -----------------------------

    window_dataset["driver"] = driver

    window_dataset["trip"] = trip

    road_type = trip.split("-")[-1]

    behavior = trip.split("-")[-2]

    window_dataset["road_type"] = road_type

    window_dataset["behavior"] = behavior

    return window_dataset

In [8]:
def create_sliding_windows_v4(
    feature_df,
    feature_list,
    window_size
):

    all_window_features = []

    for start in range(
        0,
        len(feature_df) - window_size + 1
    ):

        end = start + window_size

        window = feature_df.iloc[start:end]

        window_stats = extract_window_features_v4(
            window,
            feature_list
        )

        all_window_features.append(
            window_stats
        )

    return pd.DataFrame(
        all_window_features
    )

In [9]:
def extract_window_features_v4(window, feature_list):

    window_stats = {}

    for feature in feature_list:

        stats = extract_statistics_v4(
            window[feature],
            feature
        )

        for stat_name, stat_value in stats.items():

            column_name = f"{feature}_{stat_name}"

            window_stats[column_name] = stat_value

    return window_stats

In [10]:
def extract_statistics_v4(signal, feature_name):

    features = {}

    binary_features = [

        "vehicle_present",

        "lane_departure"

    ]

    if feature_name in binary_features:

        features["mean"] = signal.mean()

        return features

    # ----------------------------------------------------
    # Continuous Features
    # ----------------------------------------------------

    features["mean"] = signal.mean()
    features["std"] = signal.std()
    features["variance"] = signal.var()

    features["min"] = signal.min()
    features["max"] = signal.max()

    features["median"] = signal.median()

    features["rms"] = np.sqrt(
        np.mean(signal ** 2)
    )

    features["skewness"] = signal.skew()

    features["kurtosis"] = signal.kurt()

    features["q25"] = signal.quantile(0.25)

    features["q75"] = signal.quantile(0.75)

    features["iqr"] = (
        features["q75"] -
        features["q25"]
    )

    return features

In [11]:
window_features_v6 = [

    # Acceleration
    "acc_resultant",
    "acc_horizontal",

    # GPS
    "speed",
    "speed_delta",

    "roll",
    "pitch",
    "yaw",

    # Lane
    "lane_offset",
    "phi",
    "steering_alignment",
    "lane_offset_velocity",   # <-- yeni

    # Vehicle
    "front_distance",
    "relative_speed",
    "closing_speed",
    "ttc",
    "ttc_trend",              # <-- yeni

    # Binary Features
    "vehicle_present",
    "lane_departure"
]

print(window_features_v6)
print(len(window_features_v6))

['acc_resultant', 'acc_horizontal', 'speed', 'speed_delta', 'roll', 'pitch', 'yaw', 'lane_offset', 'phi', 'steering_alignment', 'lane_offset_velocity', 'front_distance', 'relative_speed', 'closing_speed', 'ttc', 'ttc_trend', 'vehicle_present', 'lane_departure']
18


In [12]:
sample_dataset = process_trip_v6(
    dataset_path,
    "D1",
    "20151110175712-16km-D1-NORMAL1-SECONDARY",
    120
)

print(sample_dataset.shape)

sample_dataset.head()

(6051, 198)


,acc_resultant_mean,acc_resultant_std,acc_resultant_variance,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_resultant_skewness,acc_resultant_kurtosis,acc_resultant_q25,...,ttc_trend_kurtosis,ttc_trend_q25,ttc_trend_q75,ttc_trend_iqr,vehicle_present_mean,lane_departure_mean,driver,trip,road_type,behavior
0,0.047805,0.020717,0.000429,0.01456,0.118106,0.043915,0.052067,0.813940,0.508880,0.031969,...,12.289562,-0.033720,0.024507,0.058227,0.791667,0.175,D1,20151110175712-16km-D1-NORMAL1-SECONDARY,SECONDARY,NORMAL1
1,0.048073,0.020655,0.000427,0.01456,0.118106,0.044198,0.052288,0.791878,0.500865,0.032249,...,12.253938,-0.035763,0.024507,0.060270,0.800000,0.175,D1,20151110175712-16km-D1-NORMAL1-SECONDARY,SECONDARY,NORMAL1
2,0.048006,0.020660,0.000427,0.01456,0.118106,0.043915,0.052229,0.800895,0.507521,0.032249,...,12.116714,-0.035763,0.032321,0.068084,0.808333,0.175,D1,20151110175712-16km-D1-NORMAL1-SECONDARY,SECONDARY,NORMAL1
3,0.047872,0.020629,0.000426,0.01456,0.118106,0.043915,0.052094,0.822268,0.549757,0.032249,...,12.096814,-0.039703,0.032321,0.072024,0.816667,0.175,D1,20151110175712-16km-D1-NORMAL1-SECONDARY,SECONDARY,NORMAL1
4,0.047977,0.020590,0.000424,0.01456,0.118106,0.044198,0.052175,0.814055,0.558273,0.032249,...,12.077815,-0.042228,0.032321,0.074549,0.816667,0.175,D1,20151110175712-16km-D1-NORMAL1-SECONDARY,SECONDARY,NORMAL1


In [13]:
trip_list = []

for driver in sorted(os.listdir(dataset_path)):

    if not driver.startswith("D"):
        continue

    driver_path = os.path.join(dataset_path, driver)

    if not os.path.isdir(driver_path):
        continue

    for trip in sorted(os.listdir(driver_path)):

        trip_path = os.path.join(driver_path, trip)

        if os.path.isdir(trip_path):

            trip_list.append({
                "driver": driver,
                "trip": trip
            })

print(len(trip_list))

40


In [14]:
from sklearn.model_selection import train_test_split

train_trips, test_trips = train_test_split(
    trip_list,
    test_size=0.20,
    random_state=42
)

print(len(train_trips))
print(len(test_trips))

32
8


In [15]:
train_datasets = []

for trip in train_trips:

    print(
        f"Processing Train : {trip['driver']} - {trip['trip']}"
    )

    trip_dataset = process_trip_v6(
        dataset_path,
        trip["driver"],
        trip["trip"],
        90
    )

    train_datasets.append(trip_dataset)

Processing Train : D6 - 20151221120051-26km-D6-AGGRESSIVE-MOTORWAY
Processing Train : D1 - 20151111135612-13km-D1-DROWSY-SECONDARY
Processing Train : D4 - 20151204152848-25km-D4-NORMAL-MOTORWAY
Processing Train : D2 - 20151120135152-25km-D2-DROWSY-MOTORWAY
Processing Train : D2 - 20151120164606-16km-D2-DROWSY-SECONDARY
Processing Train : D5 - 20151211162829-16km-D5-NORMAL1-SECONDARY
Processing Train : D5 - 20151211170502-16km-D5-DROWSY-SECONDARY
Processing Train : D2 - 20151120133502-26km-D2-AGGRESSIVE-MOTORWAY
Processing Train : D3 - 20151126125458-16km-D3-NORMAL2-SECONDARY
Processing Train : D4 - 20151203175637-17km-D4-DROWSY-SECONDARY
Processing Train : D1 - 20151110175712-16km-D1-NORMAL1-SECONDARY
Processing Train : D5 - 20151211165606-12km-D5-AGGRESSIVE-SECONDARY
Processing Train : D1 - 20151111134545-16km-D1-AGGRESSIVE-SECONDARY
Processing Train : D2 - 20151120162105-17km-D2-NORMAL2-SECONDARY
Processing Train : D1 - 20151110180824-16km-D1-NORMAL2-SECONDARY
Processing Train : D5 -

In [16]:
train_dataset = pd.concat(
    train_datasets,
    ignore_index=True
)

print(train_dataset.shape)

train_dataset.head()

(242005, 198)


,acc_resultant_mean,acc_resultant_std,acc_resultant_variance,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_resultant_skewness,acc_resultant_kurtosis,acc_resultant_q25,...,ttc_trend_kurtosis,ttc_trend_q25,ttc_trend_q75,ttc_trend_iqr,vehicle_present_mean,lane_departure_mean,driver,trip,road_type,behavior
0,0.061990,0.037068,0.001374,0.010724,0.178804,0.056313,0.072122,1.073811,0.879893,0.034824,...,30.885933,0.0,0.0,0.0,0.277778,0.211111,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE
1,0.061453,0.035682,0.001273,0.010724,0.160028,0.056313,0.070961,0.928215,0.375800,0.034824,...,30.862940,0.0,0.0,0.0,0.288889,0.211111,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE
2,0.061738,0.035498,0.001260,0.010724,0.160028,0.056313,0.071118,0.929125,0.404895,0.035379,...,30.852368,0.0,0.0,0.0,0.300000,0.211111,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE
3,0.061618,0.035434,0.001256,0.010724,0.160028,0.056313,0.070982,0.941500,0.441326,0.035379,...,29.571537,0.0,0.0,0.0,0.311111,0.211111,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE
4,0.061695,0.035540,0.001263,0.010724,0.160028,0.056313,0.071101,0.939167,0.415087,0.035379,...,27.916088,0.0,0.0,0.0,0.322222,0.211111,D6,20151221120051-26km-D6-AGGRESSIVE-MOTORWAY,MOTORWAY,AGGRESSIVE


In [17]:
test_datasets = []

for trip in test_trips:

    print(
        f"Processing Test : {trip['driver']} - {trip['trip']}"
    )

    trip_dataset = process_trip_v6(
        dataset_path,
        trip["driver"],
        trip["trip"],
        120
    )

    test_datasets.append(trip_dataset)

Processing Test : D3 - 20151126132013-17km-D3-DROWSY-SECONDARY
Processing Test : D3 - 20151126124208-16km-D3-NORMAL1-SECONDARY
Processing Test : D3 - 20151126113754-26km-D3-DROWSY-MOTORWAY
Processing Test : D4 - 20151204154908-25km-D4-AGGRESSIVE-MOTORWAY
Processing Test : D1 - 20151111132348-25km-D1-DROWSY-MOTORWAY
Processing Test : D2 - 20151120163350-16km-D2-AGGRESSIVE-SECONDARY
Processing Test : D6 - 20151221112434-17km-D6-NORMAL-SECONDARY
Processing Test : D4 - 20151204160823-25km-D4-DROWSY-MOTORWAY


In [18]:
test_dataset = pd.concat(
    test_datasets,
    ignore_index=True
)

print(test_dataset.shape)

test_dataset.head()

(65580, 198)


,acc_resultant_mean,acc_resultant_std,acc_resultant_variance,acc_resultant_min,acc_resultant_max,acc_resultant_median,acc_resultant_rms,acc_resultant_skewness,acc_resultant_kurtosis,acc_resultant_q25,...,ttc_trend_kurtosis,ttc_trend_q25,ttc_trend_q75,ttc_trend_iqr,vehicle_present_mean,lane_departure_mean,driver,trip,road_type,behavior
0,0.060652,0.036519,0.001334,0.007071,0.147425,0.05066,0.070718,1.056772,0.352917,0.035113,...,0.0,0.0,0.0,0.0,0.0,0.0,D3,20151126132013-17km-D3-DROWSY-SECONDARY,SECONDARY,DROWSY
1,0.060100,0.035696,0.001274,0.007071,0.147425,0.05066,0.069825,1.059923,0.446986,0.035113,...,0.0,0.0,0.0,0.0,0.0,0.0,D3,20151126132013-17km-D3-DROWSY-SECONDARY,SECONDARY,DROWSY
2,0.059835,0.035510,0.001261,0.007071,0.147425,0.05066,0.069503,1.086779,0.539634,0.035113,...,0.0,0.0,0.0,0.0,0.0,0.0,D3,20151126132013-17km-D3-DROWSY-SECONDARY,SECONDARY,DROWSY
3,0.059227,0.034621,0.001199,0.007071,0.147425,0.05066,0.068531,1.091184,0.650685,0.035113,...,0.0,0.0,0.0,0.0,0.0,0.0,D3,20151126132013-17km-D3-DROWSY-SECONDARY,SECONDARY,DROWSY
4,0.058609,0.033694,0.001135,0.007071,0.147425,0.05066,0.067534,1.090084,0.755289,0.035113,...,0.0,0.0,0.0,0.0,0.0,0.0,D3,20151126132013-17km-D3-DROWSY-SECONDARY,SECONDARY,DROWSY


In [19]:
train_dataset.to_csv(
    "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/processed/train_dataset_v6_window120.csv",
    index=False
)

test_dataset.to_csv(
    "/content/drive/MyDrive/TOGG_Driver_Risk_Project/data/processed/test_dataset_v6_window120.csv",
    index=False
)

print("Datasets saved successfully!")

Datasets saved successfully!
